## delete everything the setup created

In [ ]:
import os
import shutil
from pathlib import Path

BASE_DIR = Path("/content")

# Never touch a mounted Google Drive: deleting it removes your real Drive files.
KEEP = {"drive"}

# Leave the working directory first, since /content/notebooks gets deleted.
os.chdir(BASE_DIR)

for item in BASE_DIR.iterdir():
    if item.name in KEEP:
        print("Skipped:", item)
        continue

    try:
        if item.is_symlink() or item.is_file():
            item.unlink()
            print("Deleted file:", item)
        elif item.is_dir():
            shutil.rmtree(item, ignore_errors=True)
            print("Deleted folder:", item)
    except Exception as e:
        print(f"Failed to delete {item}: {e}")

print("Cleanup complete.")

## this to upload data to colab and install requerments

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

# ---------- Configuration ----------
REQ_URL = (
    "https://raw.githubusercontent.com/"
    "oelaimar/NeuroVision-AI/master/requirements.txt"
)
REPO_ZIP_URL = (
    "https://codeload.github.com/"
    "oelaimar/NeuroVision-AI/zip/refs/heads/master"
)

BASE_DIR = Path("/content")
REQ_IN = BASE_DIR / "requirements.txt"
REQ_OUT = BASE_DIR / "requirements_filtered.txt"

REPO_ZIP = BASE_DIR / "repo.zip"
REPO_TMP = BASE_DIR / "tmp_repo"
SRC_DIR = BASE_DIR / "src"

FILE_ID = "1p7R-87Embk8yJq35tkcJDpJ0f72m8-bs"
ZIP_PATH = BASE_DIR / "data.zip"
TMP_DIR = BASE_DIR / "tmp_unzip"
RAW_DIR = BASE_DIR / "data/raw"
WORK_DIR = BASE_DIR / "notebooks"

CLASSES = {"glioma", "meningioma", "notumor", "pituitary"}

SKIP = {
    "numpy", "pandas", "scipy", "matplotlib", "scikit-learn",
    "pillow", "torch", "torchvision", "tensorflow",
    "opencv-python",
}


# ---------- Helpers ----------
def run(command):
    subprocess.check_call(command)


def has_classes(path):
    """Check that every expected class contains at least one image."""
    path = Path(path)

    if not path.is_dir():
        return False

    for class_name in CLASSES:
        class_dir = path / class_name

        if not class_dir.is_dir():
            return False

        if not any(
            file.is_file()
            for file in class_dir.rglob("*")
            if file.suffix.lower() in {
                ".jpg", ".jpeg", ".png", ".bmp", ".webp"
            }
        ):
            return False

    return True


def safe_extract(archive, destination):
    """Reject ZIP entries that would extract outside destination."""
    destination = Path(destination).resolve()

    for member in archive.infolist():
        target = (destination / member.filename).resolve()

        if os.path.commonpath([str(destination), str(target)]) != str(destination):
            raise ValueError(
                f"Unsafe path in ZIP archive: {member.filename}"
            )

    archive.extractall(destination)


def install_requirements():
    print("Downloading requirements...")

    with urllib.request.urlopen(REQ_URL, timeout=30) as response:
        REQ_IN.write_bytes(response.read())

    # Normalize package names for comparison.
    skipped = {
        re.sub(r"[-_.]+", "-", name).lower()
        for name in SKIP
    }

    with REQ_IN.open(encoding="utf-8") as source, \
            REQ_OUT.open("w", encoding="utf-8") as output:

        for original_line in source:
            line = original_line.strip()

            if not line or line.startswith("#"):
                continue

            # This simple filter intentionally excludes pip options
            # and nested requirements; review these if your file uses them.
            if line.startswith("-"):
                print(f"Skipping pip directive: {line}")
                continue

            # Handles standard package specs and extras, such as
            # package>=1.0 and package[extra]>=1.0.
            match = re.match(r"^([A-Za-z0-9_.-]+)", line)

            if not match:
                print(f"Review unsupported requirement: {line}")
                continue

            name = re.sub(r"[-_.]+", "-", match.group(1)).lower()

            if name not in skipped:
                output.write(line + "\n")

    if REQ_OUT.stat().st_size:
        run([
            sys.executable, "-m", "pip", "install",
            "-r", str(REQ_OUT)
        ])

    print("Requirements processed.")


def fetch_src():
    """Download the repo ZIP and copy its src/ folder to /content/src."""
    print("Fetching src folder...")

    shutil.rmtree(REPO_TMP, ignore_errors=True)
    REPO_TMP.mkdir(parents=True, exist_ok=True)

    try:
        with urllib.request.urlopen(REPO_ZIP_URL, timeout=60) as response:
            REPO_ZIP.write_bytes(response.read())

        with zipfile.ZipFile(REPO_ZIP) as archive:
            safe_extract(archive, REPO_TMP)

        # The ZIP contains a single top-level folder: NeuroVision-AI-master/
        repo_root = next(
            (p for p in REPO_TMP.iterdir() if p.is_dir()),
            None,
        )

        if repo_root is None or not (repo_root / "src").is_dir():
            raise FileNotFoundError("No src/ folder found in the repository.")

        # Always replace so repo updates are picked up on re-run.
        shutil.rmtree(SRC_DIR, ignore_errors=True)
        shutil.copytree(repo_root / "src", SRC_DIR)

    finally:
        shutil.rmtree(REPO_TMP, ignore_errors=True)
        REPO_ZIP.unlink(missing_ok=True)

    # Make `import src...` work from notebooks.
    if str(BASE_DIR) not in sys.path:
        sys.path.insert(0, str(BASE_DIR))

    print(f"src ready: {SRC_DIR}")


def download_dataset():
    import gdown

    if ZIP_PATH.exists():
        try:
            with zipfile.ZipFile(ZIP_PATH) as archive:
                if archive.testzip() is None:
                    print("Using existing valid ZIP.")
                    return
        except zipfile.BadZipFile:
            pass

        print("Existing ZIP is invalid. Downloading again.")
        ZIP_PATH.unlink(missing_ok=True)

    result = gdown.download(
        id=FILE_ID,
        output=str(ZIP_PATH),
        quiet=False,
    )

    if not result or not ZIP_PATH.is_file():
        raise RuntimeError("Dataset download failed.")

    with zipfile.ZipFile(ZIP_PATH) as archive:
        bad_file = archive.testzip()
        if bad_file is not None:
            ZIP_PATH.unlink(missing_ok=True)
            raise RuntimeError(f"Corrupted ZIP member: {bad_file}")


def prepare_dataset():
    if has_classes(RAW_DIR):
        print(f"Dataset already ready: {RAW_DIR}")
        return

    # Do not keep a partially prepared dataset.
    shutil.rmtree(RAW_DIR, ignore_errors=True)
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)

    try:
        download_dataset()

        with zipfile.ZipFile(ZIP_PATH) as archive:
            safe_extract(archive, TMP_DIR)

        # Find the directory containing all four class folders.
        source_dir = next(
            (
                Path(root)
                for root, dirs, _ in os.walk(TMP_DIR)
                if CLASSES.issubset(set(dirs))
            ),
            None,
        )

        if source_dir is None or not has_classes(source_dir):
            raise FileNotFoundError(
                "Could not find all four valid class folders "
                "with images in the ZIP."
            )

        RAW_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(source_dir), str(RAW_DIR))

        if not has_classes(RAW_DIR):
            raise RuntimeError("Dataset validation failed after moving.")

        print(f"Dataset ready: {RAW_DIR}")

    finally:
        shutil.rmtree(TMP_DIR, ignore_errors=True)


def setup():
    install_requirements()
    fetch_src()

    # Install gdown only when it is actually needed.
    if not has_classes(RAW_DIR):
        try:
            import gdown  # noqa: F401
        except ImportError:
            run([
                sys.executable, "-m", "pip",
                "install", "gdown"
            ])

        prepare_dataset()

    WORK_DIR.mkdir(parents=True, exist_ok=True)
    os.chdir(WORK_DIR)

    print(f"Working directory: {Path.cwd()}")
    print("Setup complete.")


setup()

## check if all data are images

In [7]:
import os

image_extensions = {".jpg", ".jpeg", ".png"}
classes = ["glioma", "meningioma", "notumor", "pituitary"]

non_image = []
for c in classes:
    folder_path = os.path.join(RAW_DIR, c)
    for f in os.listdir(folder_path):
        full = os.path.join(folder_path, f)
        if os.path.isfile(full) and os.path.splitext(f)[1].lower() not in image_extensions:
            non_image.append(full)

print(len(non_image))

0


## import libraries and functions

In [8]:
# import os
# import numpy as np
# import pandas as pd
# from src.helper import load_image

import os
import sys
import importlib
import numpy as np
import pandas as pd

if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# Drop cached copies of src so the freshly downloaded files are imported
for name in list(sys.modules):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]
importlib.invalidate_caches()

sys.path.append('/content/')
from src.helper import load_dataset  # should print /content/src/helper.py



## import images with labels

In [9]:
data = dict()

folder_path = os.path.join(RAW_DIR)
X, y = load_dataset(folder_path, classes)
print(X.shape, X.dtype)   # (N, 224, 224, 3) uint8
print(y.shape, y.dtype)   # (N,) int64
print(np.unique(y))       # [0 1 2 3]

Successfully loaded: 1621 images
Skipped: 0 files
Successfully loaded: 1645 images
Skipped: 0 files
Successfully loaded: 2000 images
Skipped: 0 files
Successfully loaded: 1757 images
Skipped: 0 files
(7023, 224, 224, 3) uint8
(7023,) int64
[0 1 2 3]


##

In [27]:
from src.pipeline import get_datasets

train_ds, val_ds = get_datasets(RAW_DIR, CLASSES, batch_size=32)

for xb, yb in train_ds.take(1):
    print(xb.shape, xb.dtype, yb.shape)